In [ ]:
# Libraries import
import os
import shutil
import random
import numpy as np
import tensorflow as tf
import hashlib

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

# SEED
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


# PATHS
original_data = "data/train"

clean_data = "data/clean_data"
train_dir = "data/train_new"
val_dir   = "data/val"
test_dir  = "data/test_new"

# CLEAN OLD OUTPUTS
for folder in [clean_data, train_dir, val_dir, test_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)


# REMOVE DUPLICATES
print("Removing duplicate images...")

def get_hash(file_path):
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

seen_hashes = set()
removed = 0

for class_name in os.listdir(original_data):
    class_path = os.path.join(original_data, class_name)

    if not os.path.isdir(class_path):
        continue

    save_class_path = os.path.join(clean_data, class_name)
    os.makedirs(save_class_path, exist_ok=True)

    for img in os.listdir(class_path):
        img_path = os.path.join(class_path, img)
        h = get_hash(img_path)

        if h not in seen_hashes:
            seen_hashes.add(h)
            shutil.copy(img_path, os.path.join(save_class_path, img))
        else:
            removed += 1

print(f"Duplicates removed: {removed}")


# DATA SPLIT
train_ratio = 0.7
val_ratio = 0.15

for class_name in os.listdir(clean_data):
    class_path = os.path.join(clean_data, class_name)

    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    train_imgs = images[:train_end]
    val_imgs   = images[train_end:val_end]
    test_imgs  = images[val_end:]

    for folder in [train_dir, val_dir, test_dir]:
        os.makedirs(os.path.join(folder, class_name), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(train_dir, class_name, img))

    for img in val_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(val_dir, class_name, img))

    for img in test_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(test_dir, class_name, img))

print("spliting train data completed")


# DATA AUGMENTATION
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)


# LOAD DATA
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

# CLASS DISTRIBUTION
def print_class_distribution(generator, name):
    classes = generator.classes
    unique, counts = np.unique(classes, return_counts=True)

    print(f"\n📊 {name} CLASS DISTRIBUTION:")
    for cls, count in zip(unique, counts):
        label = list(generator.class_indices.keys())[list(generator.class_indices.values()).index(cls)]
        print(f"{label}: {count}")

    return dict(zip(unique, counts))

train_before = print_class_distribution(train_gen, "TRAIN (BEFORE BALANCE)")
val_before   = print_class_distribution(val_gen, "VALIDATION")
test_before  = print_class_distribution(test_gen, "TEST")

# CLASS WEIGHTS
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)

class_weights = dict(enumerate(class_weights))

print("\n⚖️ CLASS WEIGHTS:")
print(class_weights)

# MODEL 
model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(32, 3, padding='same', activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.SpatialDropout2D(0.15),

    layers.Conv2D(64, 3, padding='same', activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.SpatialDropout2D(0.20),

    layers.Conv2D(128, 3, padding='same', activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.SpatialDropout2D(0.25),

    layers.Conv2D(256, 3, padding='same', activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.SpatialDropout2D(0.30),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

# COMPILE =
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# CALLBACKS
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    ModelCheckpoint("best_model.keras", save_best_only=True),
    reduce_lr
]

# TRAINING
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks
)


# EVALUATION
print("Evaluation og model")
loss, acc = model.evaluate(test_gen)
print(f"Test Accuracy: {acc*100:.2f}%")

# PREDICTIONS
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes

print("\nFinal Accuracy:", np.mean(y_pred == y_true))

Removing duplicate images...
Duplicates removed: 3
spliting train data completed
Found 1344 images belonging to 2 classes.
Found 288 images belonging to 2 classes.
Found 289 images belonging to 2 classes.

📊 TRAIN (BEFORE BALANCE) CLASS DISTRIBUTION:
infected: 545
notinfected: 799

📊 VALIDATION CLASS DISTRIBUTION:
infected: 117
notinfected: 171

📊 TEST CLASS DISTRIBUTION:
infected: 117
notinfected: 172

⚖️ CLASS WEIGHTS:
{0: np.float64(1.2330275229357799), 1: np.float64(0.8410513141426783)}


c:\Users\91850\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 209s 5s/step - accuracy: 0.5975 - loss: 0.8319 - val_accuracy: 0.4410 - val_loss: 0.6575 - learning_rate: 5.0000e-05
Epoch 2/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 184s 4s/step - accuracy: 0.6704 - loss: 0.6693 - val_accuracy: 0.8264 - val_loss: 0.5883 - learning_rate: 5.0000e-05
Epoch 3/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 173s 4s/step - accuracy: 0.7009 - loss: 0.6252 - val_accuracy: 0.6701 - val_loss: 0.5430 - learning_rate: 5.0000e-05
Epoch 4/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 195s 5s/step - accuracy: 0.7507 - loss: 0.5857 - val_accuracy: 0.6042 - val_loss: 0.5231 - learning_rate: 5.0000e-05
Epoch 5/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 199s 5s/step - accuracy: 0.7656 - loss: 0.5010 - val_accuracy: 0.6319 - val_loss: 0.5022 - learning_rate: 5.0000e-05
Epoch 6/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 203s 5s/step - accuracy: 0.7604 - loss: 0.4948 - val_accuracy: 0.6319 - val_loss: 0.4973 - learning_rate: 5.0000e-05
Epoch 7/20
42/42 ━━━━━━━━━━━━━━━━━━━━ 185s 4s/step - accuracy: 0.7790 

In [33]:

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd

#predictions
y_pred_prob = model.predict(test_gen)
y_pred = (y_pred_prob > 0.7).astype(int).flatten()
y_true = test_gen.classes


#confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("CONFUSION MATRIX:")
print(cm)


# CLASSIFICATION REPORT
print("CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=list(test_gen.class_indices.keys())))


# ACCURACY + MISMATCH
total = len(y_true)
correct = np.sum(y_pred == y_true)
wrong = total - correct

print("FINAL RESULTS:")
print(f"Total Test Images: {total}")
print(f"Correct Predictions: {correct}")
print(f"Wrong Predictions: {wrong}")
print(f"Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"Mismatch Rate: {(wrong/total)*100:.2f}%")


# MISCLASSIFIED IMAGES 

filenames = test_gen.filenames

class_names = list(test_gen.class_indices.keys())

data = []

for i in range(len(y_true)):
    if y_pred[i] != y_true[i]:
        data.append([
            filenames[i],
            class_names[y_true[i]],
            class_names[y_pred[i]],
            float(y_pred_prob[i][0])
        ])


df = pd.DataFrame(data, columns=[
    "Image_Path",
    "Actual_Label",
    "Predicted_Label",
    "Prediction_Confidence"
])

#SAVE TO CSV
csv_path = "misclassified_images.csv"
df.to_csv(csv_path, index=False)

print(f"\n✅ Misclassified images saved to: {csv_path}")
print(f"Total mismatches: {len(df)}")


10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 547ms/step
CONFUSION MATRIX:
[[104  13]
 [  0 172]]
CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    infected       1.00      0.89      0.94       117
 notinfected       0.93      1.00      0.96       172

    accuracy                           0.96       289
   macro avg       0.96      0.94      0.95       289
weighted avg       0.96      0.96      0.95       289

FINAL RESULTS:
Total Test Images: 289
Correct Predictions: 276
Wrong Predictions: 13
Accuracy: 95.50%
Mismatch Rate: 4.50%

✅ Misclassified images saved to: misclassified_images.csv
Total mismatches: 13


In [32]:
for t in [0.3, 0.5, 0.7, 0.85]:
    y_pred = (y_pred_prob > t).astype(int)
    print(t, accuracy_score(y_true, y_pred))

0.3 0.8373702422145328
0.5 0.8927335640138409
0.7 0.9550173010380623
0.85 0.9965397923875432


In [42]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# =========================
# LOAD MODEL
# =========================
model = tf.keras.models.load_model("best_model.keras")

# =========================
# SETTINGS (SAME AS STREAMLIT)
# =========================
IMG_SIZE = (224, 224)
THRESHOLD = 0.70

# IMPORTANT: match your class_indices
CLASS_MAP = {0: "Infected", 1: "Not Infected"}

test_dir = "data/test_new"

# =========================
# STORE RESULTS
# =========================
results = []
y_true = []
y_pred = []

# =========================
# LOOP THROUGH TEST DATA
# =========================
for class_name in os.listdir(test_dir):
    class_path = os.path.join(test_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    true_label = 0 if class_name.lower() == "infected" else 1

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        try:
            # =========================
            # SAME PREPROCESSING AS STREAMLIT
            # =========================
            image = Image.open(img_path).convert("RGB")
            image = image.resize(IMG_SIZE)

            img_array = np.array(image).astype("float32") / 255.0
            img_array = np.expand_dims(img_array, axis=0)

            # =========================
            # PREDICTION
            # =========================
            prob = model.predict(img_array, verbose=0)[0][0]
            pred_label = 1 if prob > THRESHOLD else 0

            # Store for metrics
            y_true.append(true_label)
            y_pred.append(pred_label)

            # Save result
            results.append({
                "Image_Path": img_path,
                "Actual_Label": CLASS_MAP[true_label],
                "Predicted_Label": CLASS_MAP[pred_label],
                "Probability": float(prob),
                "Result": "Correct" if true_label == pred_label else "Wrong"
            })

        except Exception as e:
            print(f"Error processing {img_path}: {e}")

# =========================
# CONVERT TO DATAFRAME
# =========================
df = pd.DataFrame(results)

# =========================
# SAVE CSV FILES
# =========================
df.to_csv("streamlit_style_all_predictions.csv", index=False)
df[df["Result"] == "Wrong"].to_csv("streamlit_style_misclassified.csv", index=False)

print("✅ Saved:")
print("→ streamlit_style_all_predictions.csv")
print("→ streamlit_style_misclassified.csv")

# =========================
# EVALUATION
# =========================
cm = confusion_matrix(y_true, y_pred)

print("\n📊 CONFUSION MATRIX:")
print(cm)

print("\n📋 CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=["Infected", "Not Infected"]))

total = len(y_true)
correct = np.sum(np.array(y_true) == np.array(y_pred))
wrong = total - correct

print("\n📊 FINAL RESULTS:")
print(f"Total: {total}")
print(f"Correct: {correct}")
print(f"Wrong: {wrong}")
print(f"Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"Mismatch Rate: {(wrong/total)*100:.2f}%")

✅ Saved:
→ streamlit_style_all_predictions.csv
→ streamlit_style_misclassified.csv

📊 CONFUSION MATRIX:
[[ 95  22]
 [  0 172]]

📋 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Infected       1.00      0.81      0.90       117
Not Infected       0.89      1.00      0.94       172

    accuracy                           0.92       289
   macro avg       0.94      0.91      0.92       289
weighted avg       0.93      0.92      0.92       289


📊 FINAL RESULTS:
Total: 289
Correct: 267
Wrong: 22
Accuracy: 92.39%
Mismatch Rate: 7.61%
